# Data Cleaning — BRFSS 2015 (`_MICHD`)

Two stages:
1. **Stateless cleaning** (`preprocess.py`) — codebook rules, safe to apply to train + test at once
2. **Learned cleaning** (`fit` / `transform`) — imputation, NaN indicators, clipping, one-hot encoding, standardisation — fit on train only

Output: `data/clean/*.npy` ready for `run.ipynb`

In [7]:
import numpy as np
import os
from helpers import load_csv_data
from preprocess import stateless_clean, leftover_codes
from learned_preprocess import fit, transform

## 1. Load raw data

In [8]:
RAW_DIR   = "data/raw/"
CLEAN_DIR = "data/clean/"

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(RAW_DIR)

# Read column names from the CSV header
with open(os.path.join(RAW_DIR, "x_train.csv")) as f:
    header = f.readline().strip().split(",")
full_headers = header[1:] if len(header) == x_train.shape[1] + 1 else header

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print(f"Positives: {(y_train == 1).sum()}  Negatives: {(y_train == -1).sum()}")

x_train: (328135, 321)
x_test : (109379, 321)
Positives: 28975  Negatives: 299160


## 2. Stateless cleaning (codebook rules)

All rules come from `preprocess.py` — nothing is learned from the data here.

In [9]:
x_tr_c, clean_names, rep = stateless_clean(x_train, full_headers)
x_te_c, te_names, _     = stateless_clean(x_test,  full_headers)

assert clean_names == te_names, "train and test columns differ after cleaning"

idx = {n: j for j, n in enumerate(clean_names)}

print(f"Columns: {x_train.shape[1]} → {x_tr_c.shape[1]}  (dropped {x_train.shape[1] - x_tr_c.shape[1]})")
print(f"NaN share: {np.isnan(x_train).mean():.3f} → {np.isnan(x_tr_c).mean():.3f}")
print("Rules not found:", rep["rules_not_found"])
print("Unverified NaN share:", rep["unverified_nan_share"])

left = leftover_codes(x_tr_c, clean_names)
print("\nLeftover special codes in mostly-filled columns:")
for item in left:
    print(" ", item)
if not left:
    print("  none")

Columns: 321 → 275  (dropped 46)
NaN share: 0.448 → 0.449
Rules not found: []
Unverified NaN share: {'MAXVO2_': 0.012, 'FC60_': 0.012, 'DROCDY3_': 0.0487}

Leftover special codes in mostly-filled columns:
  ('_AGE80', 77, 0.013)


## 3. Learned cleaning: `fit` / `transform`

Everything estimated from data (medians, quantiles, one-hot categories) is fit on the **training set only**, then applied to both train and test.

In [10]:
# fit/transform are imported from learned_preprocess.py
# See that file for the full implementation and documentation.

## 4. Sanity check with one fold

In [11]:
n = len(y_train)
rng = np.random.default_rng(42)
perm = rng.permutation(n)
tr_idx = perm[:int(0.8 * n)]
va_idx = perm[int(0.8 * n):]

params = fit(x_tr_c[tr_idx], clean_names)
X_tr, feat_names = transform(x_tr_c[tr_idx], params)
X_va, _          = transform(x_tr_c[va_idx], params)

print("X_train:", X_tr.shape, "  dtype:", X_tr.dtype)
print("X_val  :", X_va.shape)
print("NaN in train:", np.isnan(X_tr).sum(), " val:", np.isnan(X_va).sum())
print(f"Features: {len(feat_names)}  (first 5: {feat_names[:5]})")

X_train: (262508, 369)   dtype: float32
X_val  : (65627, 369)
NaN in train: 0  val: 0
Features: 369  (first 5: ['bias', 'NUMADULT_isnan', 'NUMADULT', 'NUMMEN_isnan', 'NUMMEN'])


## 5. Save clean data

In [12]:
os.makedirs(CLEAN_DIR, exist_ok=True)

np.save(CLEAN_DIR + "x_train_clean.npy", x_tr_c)
np.save(CLEAN_DIR + "x_test_clean.npy",  x_te_c)
np.save(CLEAN_DIR + "y_train.npy",        y_train)
np.save(CLEAN_DIR + "train_ids.npy",      train_ids)
np.save(CLEAN_DIR + "test_ids.npy",       test_ids)

with open(CLEAN_DIR + "columns.txt", "w") as f:
    f.write("\n".join(clean_names))

print("Saved to", CLEAN_DIR)
print("x_train_clean:", x_tr_c.shape)
print("x_test_clean :", x_te_c.shape)

Saved to data/clean/
x_train_clean: (328135, 275)
x_test_clean : (109379, 275)
